# Группа 2. Три светофора

Три выезда находятся на одинаковом удалении. Нужно выбрать светофор, к которому лучше ехать через час. Запустите ячейку, сравните baseline, линейную модель и MLP, затем измените параметры генератора.


In [ ]:
"""
Практикум группы 2.

Три светофора находятся на одинаковом удалении от точки старта.
По истории задержек нужно выбрать, к какому светофору лучше ехать через час.

Запуск:
    python3 "Группа 2 — три светофора.py"

Результаты сохраняются в папке outputs_traffic_lights.
"""

from pathlib import Path
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


# ПАРАМЕТРЫ, КОТОРЫЕ МОЖНО МЕНЯТЬ
SEED = 42
DAYS = 28
STEP_MINUTES = 15
FORECAST_MINUTES = 60
TRAIN_SHARE = 0.60
VALIDATION_SHARE = 0.20
NOISE_LEVEL = 0.9
INCIDENT_PROBABILITY = 0.012
RUN_MLP = True
RUN_LEAKAGE_DEMO = True
RUN_FINAL_TEST = False  # Включить один раз после выбора модели по validation.


RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path.cwd() / "outputs_traffic_lights" / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def peak(hour, center, width):
    """Гладкий пик транспортной нагрузки."""
    return np.exp(-0.5 * ((hour - center) / width) ** 2)


def generate_data():
    rng = np.random.default_rng(SEED)
    periods = DAYS * 24 * 60 // STEP_MINUTES
    time = pd.date_range("2026-01-01", periods=periods, freq=f"{STEP_MINUTES}min")
    hour = time.hour.to_numpy() + time.minute.to_numpy() / 60
    weekday = time.dayofweek.to_numpy()
    workday = (weekday < 5).astype(float)

    morning = peak(hour, 8.0, 1.4) * workday
    evening = peak(hour, 18.0, 1.8) * workday
    weekend = (weekday >= 5).astype(float)
    shopping = peak(hour, 14.0, 2.4) * weekend

    # У светофоров разные профили. Расстояние одинаковое, различается задержка.
    raw_a = 3.5 + 8.5 * morning + 2.0 * evening + 1.5 * shopping
    raw_b = 4.5 + 3.0 * morning + 7.0 * evening + 2.5 * shopping
    raw_c = 5.3 + 4.2 * morning + 4.5 * evening + 1.0 * shopping

    delays = {}
    for name, raw in {"A": raw_a, "B": raw_b, "C": raw_c}.items():
        values = np.zeros(periods)
        values[0] = raw[0] + rng.normal(0, NOISE_LEVEL)
        for i in range(1, periods):
            incident = 0.0
            if rng.random() < INCIDENT_PROBABILITY:
                incident = rng.uniform(4, 12)
            # Инерция: текущая задержка зависит от предыдущего состояния.
            values[i] = 0.72 * values[i - 1] + 0.28 * raw[i] + rng.normal(0, NOISE_LEVEL) + incident
            values[i] = max(values[i], 0.5)
        delays[name] = values

    data = pd.DataFrame(
        {
            "time": time,
            "hour": hour,
            "weekday": weekday,
            "delay_A": delays["A"],
            "delay_B": delays["B"],
            "delay_C": delays["C"],
        }
    )
    return data


def build_supervised_table(data):
    horizon_steps = FORECAST_MINUTES // STEP_MINUTES
    if horizon_steps < 1:
        raise ValueError("FORECAST_MINUTES должен быть не меньше STEP_MINUTES")

    table = data.copy()
    table["hour_sin"] = np.sin(2 * np.pi * table["hour"] / 24)
    table["hour_cos"] = np.cos(2 * np.pi * table["hour"] / 24)
    table["weekday_sin"] = np.sin(2 * np.pi * table["weekday"] / 7)
    table["weekday_cos"] = np.cos(2 * np.pi * table["weekday"] / 7)

    for light in "ABC":
        for lag in (1, 4, 8):
            table[f"delay_{light}_lag_{lag}"] = table[f"delay_{light}"].shift(lag)
        table[f"target_{light}"] = table[f"delay_{light}"].shift(-horizon_steps)

    feature_columns = [
        "hour_sin",
        "hour_cos",
        "weekday_sin",
        "weekday_cos",
        "delay_A",
        "delay_B",
        "delay_C",
    ]
    feature_columns += [f"delay_{light}_lag_{lag}" for light in "ABC" for lag in (1, 4, 8)]
    target_columns = [f"target_{light}" for light in "ABC"]

    table = table.dropna().reset_index(drop=True)
    return table, feature_columns, target_columns


def decision_metrics(actual, predicted):
    actual_choice = np.argmin(actual, axis=1)
    predicted_choice = np.argmin(predicted, axis=1)
    accuracy = np.mean(actual_choice == predicted_choice)
    chosen_actual = actual[np.arange(len(actual)), predicted_choice]
    best_actual = np.min(actual, axis=1)
    regret = np.mean(chosen_actual - best_actual)
    return accuracy, regret


def evaluate(name, actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    choice_accuracy, regret = decision_metrics(actual, predicted)
    return {
        "model": name,
        "MAE_minutes": round(float(mae), 3),
        "correct_light_share": round(float(choice_accuracy), 3),
        "mean_extra_wait_minutes": round(float(regret), 3),
    }


data = generate_data()
table, feature_columns, target_columns = build_supervised_table(data)

if TRAIN_SHARE + VALIDATION_SHARE >= 1:
    raise ValueError("TRAIN_SHARE + VALIDATION_SHARE должны быть меньше 1")

# Разрыв на горизонт прогноза не даёт целевым значениям одной части попасть
# во временной интервал следующей части.
horizon_steps = FORECAST_MINUTES // STEP_MINUTES
train_end = int(len(table) * TRAIN_SHARE)
validation_end = int(len(table) * (TRAIN_SHARE + VALIDATION_SHARE))
train = table.iloc[: train_end - horizon_steps].copy()
validation = table.iloc[train_end : validation_end - horizon_steps].copy()
test = table.iloc[validation_end:].copy()

X_train = train[feature_columns]
X_validation = validation[feature_columns]
y_train = train[target_columns].to_numpy()
y_validation = validation[target_columns].to_numpy()

# Baseline: через час задержки будут такими же, как сейчас.
baseline_prediction = validation[["delay_A", "delay_B", "delay_C"]].to_numpy()

ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
ridge.fit(X_train, y_train)
ridge_prediction = ridge.predict(X_validation)

results = [evaluate("Последнее значение", y_validation, baseline_prediction)]
results.append(evaluate("Линейная Ridge", y_validation, ridge_prediction))

predictions = {"Baseline": baseline_prediction, "Ridge": ridge_prediction}

if RUN_MLP:
    mlp = make_pipeline(
        StandardScaler(),
        MLPRegressor(
            hidden_layer_sizes=(32, 16),
            random_state=SEED,
            max_iter=350,
            early_stopping=True,
        ),
    )
    mlp.fit(X_train, y_train)
    mlp_prediction = mlp.predict(X_validation)
    predictions["MLP"] = mlp_prediction
    results.append(evaluate("Небольшой MLP", y_validation, mlp_prediction))

results_table = pd.DataFrame(results)
results_table.insert(0, "split", "validation")
print("\nРЕЗУЛЬТАТЫ НА VALIDATION: ПО НИМ ВЫБИРАЕМ МОДЕЛЬ")
print(results_table.to_string(index=False))
results_table.to_csv(OUTPUT_DIR / "metrics.csv", index=False)

# Главная метрика — цена неправильного маршрутного решения. MAE дополнительна.
best_model_name = min(
    results,
    key=lambda row: (
        row["mean_extra_wait_minutes"],
        -row["correct_light_share"],
        row["MAE_minutes"],
    ),
)["model"]
best_key = {"Последнее значение": "Baseline", "Линейная Ridge": "Ridge", "Небольшой MLP": "MLP"}[best_model_name]
best_prediction = predictions[best_key]

# График непрерывного validation-участка.
show = min(7 * 24 * 60 // STEP_MINUTES, len(validation))
time_validation = validation["time"].iloc[:show]

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
for i, light in enumerate("ABC"):
    axes[i].plot(time_validation, y_validation[:show, i], label=f"Факт {light}", linewidth=2)
    axes[i].plot(time_validation, best_prediction[:show, i], label=f"Прогноз {light}", alpha=0.8)
    axes[i].set_ylabel("Минуты")
    axes[i].legend()
axes[-1].set_xlabel("Время")
fig.suptitle(f"Прогноз задержки через {FORECAST_MINUTES} минут: {best_model_name}")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "forecast.png", dpi=150)
plt.close(fig)

# Какой светофор модель выбирает на первых 20 тестовых моментах.
choice_names = np.array(["A", "B", "C"])
choice_table = pd.DataFrame(
    {
        "time": validation["time"].iloc[:20].to_numpy(),
        "model_choice": choice_names[np.argmin(best_prediction[:20], axis=1)],
        "actual_best": choice_names[np.argmin(y_validation[:20], axis=1)],
    }
)
print("\nПЕРВЫЕ 20 РЕШЕНИЙ")
print(choice_table.to_string(index=False))
choice_table.to_csv(OUTPUT_DIR / "first_20_decisions.csv", index=False)

if RUN_LEAKAGE_DEMO:
    # Намеренно неправильный эксперимент: будущие ответы подаются в признаки.
    leak_features = feature_columns + target_columns
    leak_model = make_pipeline(StandardScaler(), Ridge(alpha=0.001))
    leak_model.fit(train[leak_features], y_train)
    leak_prediction = leak_model.predict(validation[leak_features])
    leak_result = evaluate("Модель с утечкой будущего", y_validation, leak_prediction)
    print("\nНАМЕРЕННО НЕПРАВИЛЬНЫЙ ЭКСПЕРИМЕНТ")
    print(pd.DataFrame([leak_result]).to_string(index=False))
    print("Почему этот результат нельзя использовать? Найдите запрещённые признаки.")

if RUN_FINAL_TEST:
    # Test открываем один раз после выбора модели и настроек на validation.
    development = table.iloc[: validation_end - horizon_steps].copy()
    X_development = development[feature_columns]
    y_development = development[target_columns].to_numpy()
    X_test = test[feature_columns]
    y_test = test[target_columns].to_numpy()

    if best_key == "Baseline":
        final_prediction = test[["delay_A", "delay_B", "delay_C"]].to_numpy()
    elif best_key == "Ridge":
        final_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
        final_model.fit(X_development, y_development)
        final_prediction = final_model.predict(X_test)
    else:
        final_model = make_pipeline(
            StandardScaler(),
            MLPRegressor(
                hidden_layer_sizes=(32, 16),
                random_state=SEED,
                max_iter=350,
                early_stopping=True,
            ),
        )
        final_model.fit(X_development, y_development)
        final_prediction = final_model.predict(X_test)

    final_result = evaluate(best_model_name, y_test, final_prediction)
    final_result["split"] = "test"
    final_table = pd.DataFrame([final_result])[
        ["split", "model", "MAE_minutes", "correct_light_share", "mean_extra_wait_minutes"]
    ]
    print("\nФИНАЛЬНЫЙ TEST: НЕ ИСПОЛЬЗУЙТЕ ЕГО ДЛЯ НОВОГО ПОДБОРА")
    print(final_table.to_string(index=False))
    final_table.to_csv(OUTPUT_DIR / "final_test_metrics.csv", index=False)

print("\nЗАДАНИЕ ГРУППЫ")
print("1. Объясните, что означают три метрики.")
print("2. Выберите модель для первого эксперимента и обоснуйте выбор.")
print("3. Меняйте FORECAST_MINUTES и NOISE_LEVEL, сравнивая только validation.")
print("4. Объясните, почему эксперимент с утечкой будущего нечестный.")
print("5. После выбора модели верните лучшие настройки и один раз включите RUN_FINAL_TEST.")
print(f"\nФайлы результата: {OUTPUT_DIR.resolve()}")


## Что сдаёт группа

1. Таблицу результатов.
2. Один основной график.
3. Выбранную первую модель.
4. Главный риск данных.
5. Следующий эксперимент.
